In [ ]:
#@title Research methodology 4 — foreground-gated lifting on SA-VM (about 75 min on an A100)
# Runtime > Change runtime type > A100 GPU. Outputs (checkpoints, logs, REPORT.md, for_claude.zip) go to Google Drive.
# Re-running continues where it stopped. Any setting can be overridden before `%run -i`.
import os, subprocess, sys
g = globals()
RUNS       = g.get('RUNS', 'A2x:80085 FGx:80085')  # session 1: SA-VM and SA-VM + foreground gating, same seed
RESULTS    = g.get('RESULTS', '/content/drive/MyDrive/retrack_results_m2')  # outputs go to {RESULTS}/m4
DATA_CACHE = g.get('DATA_CACHE', '/content/drive/MyDrive/datasets/wildtrack.tar')  # saved by attempt 1
BRANCH     = g.get('BRANCH', 'feat/research-methodology-2')
EXTRA      = g.get('EXTRA', '')             # extra m2_pipeline.py arguments
REPO_DIR   = '/content/retrack'
OUT        = f'{RESULTS}/m4'
BG         = f'{OUT}/fg_background.npz'

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')

from google.colab import drive
drive.mount('/content/drive')

# 1. code (already cloned by the launcher cell; cloned here when this notebook is opened on its own)
if not os.path.exists(REPO_DIR):
    sh(f'git clone -q -b {BRANCH} https://github.com/rizwanmustafa/retrack.git {REPO_DIR}')
sh(f'git -C {REPO_DIR} log --oneline -1')

# 2. WILDTRACK: this session -> Drive copy -> Kaggle (then saved to Drive for next time)
W = '/content/data/Wildtrack'
have = lambda: all(os.path.exists(f'{W}/{d}') for d in ('Image_subsets', 'annotations_positions', 'calibrations'))
if not have() and DATA_CACHE and os.path.exists(DATA_CACHE):
    print('restoring WILDTRACK from Drive ...')
    os.makedirs('/content/data', exist_ok=True)
    sh(f'tar -xf {DATA_CACHE} -C /content/data')
if not have():
    print('downloading WILDTRACK from Kaggle ...')
    sh('pip -q install kagglehub')
    sh(f'cd {REPO_DIR}/WorldTrack && python -c "from retrack_pipeline import prepare_wildtrack; '
       f'prepare_wildtrack(\'/content/data\')"')
    if DATA_CACHE:
        os.makedirs(os.path.dirname(DATA_CACHE), exist_ok=True)
        sh(f'cd /content/data && tar -chf {DATA_CACHE} Wildtrack')
assert have(), 'WILDTRACK is not ready'

# 3. dependencies
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
os.makedirs(OUT, exist_ok=True)

# 4. background images from the training frames (~3 min, once; kept on Drive) and the mask check
sh(f'cd {REPO_DIR} && python research-methodology-4/code/fg_background.py --data_dir {W} --out {BG} '
   f'2>&1 | tee -a {OUT}/pipeline.log; exit ${{PIPESTATUS[0]}}')
from IPython.display import Image as _Img, display
display(_Img(f'{OUT}/fg_preview.jpg', width=720))

# 5. preflight, training, evaluation, robustness of the new checkpoints, report
sh(f'cd {REPO_DIR} && PYTHONUNBUFFERED=1 python research-methodology-2/code/m2_pipeline.py --results {RESULTS} '
   f'--subdir m4 --fg_background {BG} --attempt1 none --runs {RUNS} {EXTRA} '
   f'2>&1 | tee -a {OUT}/pipeline.log; exit ${{PIPESTATUS[0]}}')

from IPython.display import Markdown
display(Markdown(open(f'{OUT}/REPORT.md').read()))
print(f'Share {OUT}/for_claude.zip for the analysis.')
